# Comparação entre CAR e MapBiomas na bacia do rio Gramame

Este caderno executa e documenta o núcleo metodológico do TCC: auditoria dos dados do Cadastro Ambiental Rural, recorte pela bacia do rio Gramame, cálculo de áreas sem dupla contagem e comparação da vegetação nativa declarada com as classes naturais do MapBiomas 2025.

A comparação é exploratória. Divergências entre as bases não constituem validação ou invalidação jurídica dos cadastros.

## 1 Preparação do ambiente

O código analítico está no pacote `gramame_car_mapbiomas`. O notebook apresenta cada etapa, enquanto o pacote mantém as funções testáveis e reutilizáveis para outras bacias ou para todo o estado.

In [ ]:
# Preparação automática no Google Colab; em Jupyter local, esta célula não altera o ambiente.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    GITHUB_USER = 'Emanuel2806PB'

    REPOSITORY_URL = f'https://github.com/{GITHUB_USER}/gramame-car-mapbiomas.git'
    REPOSITORY_DIR = Path('/content/gramame-car-mapbiomas')
    if not REPOSITORY_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)

    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPOSITORY_DIR)], check=True)
    os.chdir(REPOSITORY_DIR)

    from google.colab import drive
    drive.mount('/content/drive')
    print('Repositório e Google Drive preparados para a análise.')
else:
    print('Ambiente local detectado; nenhuma instalação automática foi executada.')

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as NotebookImage, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() in {'analise', 'notebooks'}:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from gramame_car_mapbiomas.config import load_config
from gramame_car_mapbiomas.pipeline import _exclude_registration_statuses, run_pipeline
from gramame_car_mapbiomas.raster import summarize_classes_in_geometry
from gramame_car_mapbiomas.vector import (
    clean_vector,
    clip_to_study_area,
    dissolve_study_area,
    geometry_union,
    read_vector,
    vector_indicators,
)

pd.set_option('display.max_columns', 100)
plt.rcParams.update({'figure.figsize': (10, 7), 'axes.grid': False})
print(f'Raiz do projeto: {PROJECT_ROOT}')

## 2 Configuração e inventário das entradas

Crie `config/gramame.yaml` a partir do arquivo de exemplo e ajuste os caminhos e nomes das camadas. Enquanto a configuração definitiva não existir, o notebook lê `gramame.example.yaml` apenas para mostrar as entradas esperadas.

In [ ]:
CONFIG_LOCAL = PROJECT_ROOT / 'config' / 'gramame.yaml'
CONFIG_COLAB = PROJECT_ROOT / 'config' / 'gramame.colab.yaml'
CONFIG_COLAB_EXEMPLO = PROJECT_ROOT / 'config' / 'gramame.colab.example.yaml'
CONFIG_EXEMPLO = PROJECT_ROOT / 'config' / 'gramame.example.yaml'
if IN_COLAB:
    CONFIG_PATH = CONFIG_COLAB if CONFIG_COLAB.exists() else CONFIG_COLAB_EXEMPLO
else:
    CONFIG_PATH = CONFIG_LOCAL if CONFIG_LOCAL.exists() else CONFIG_EXEMPLO

cfg = load_config(CONFIG_PATH)
entradas = pd.DataFrame([
    {'entrada': 'Limite da bacia', 'caminho': cfg.study_area.path, 'camada': cfg.study_area.layer},
    {'entrada': 'Imóveis rurais', 'caminho': cfg.properties.path, 'camada': cfg.properties.layer},
    {'entrada': 'Vegetação declarada', 'caminho': cfg.declared_native.path, 'camada': cfg.declared_native.layer},
    {'entrada': 'MapBiomas', 'caminho': cfg.mapbiomas.path, 'camada': None},
])
entradas['existe'] = entradas['caminho'].map(Path.exists)
display(entradas)

DADOS_DISPONIVEIS = bool(entradas['existe'].all())
if DADOS_DISPONIVEIS:
    print('Todas as entradas foram encontradas. O processamento real pode ser executado.')
else:
    faltantes = entradas.loc[~entradas['existe'], 'entrada'].tolist()
    print('Entradas ainda ausentes:', ', '.join(faltantes))
    print('Ajuste config/gramame.yaml antes de executar as etapas com dados reais.')

## 3 Leitura e auditoria geométrica

Nesta etapa são conferidos CRS, geometrias ausentes, vazias ou inválidas, duplicidades geométricas e polígonos menores que o limite configurado. Os reparos são realizados em cópias de trabalho.

In [ ]:
if DADOS_DISPONIVEIS:
    area_bruta = read_vector(cfg.study_area, 'area_estudo')
    imoveis_brutos = read_vector(cfg.properties, 'imoveis')
    vegetacao_bruta = read_vector(cfg.declared_native, 'vegetacao_declarada')
    imoveis_brutos = _exclude_registration_statuses(
        imoveis_brutos, cfg.properties.status_field,
        cfg.analysis.excluded_status_codes, 'imoveis',
    )
    vegetacao_bruta = _exclude_registration_statuses(
        vegetacao_bruta, cfg.declared_native.status_field,
        cfg.analysis.excluded_status_codes, 'vegetacao_declarada',
    )

    area_limpa, qc_area = clean_vector(
        area_bruta, name='area_estudo', target_crs=cfg.project.target_crs,
        minimum_area_m2=cfg.analysis.minimum_geometry_area_m2,
    )
    imoveis_limpos, qc_imoveis = clean_vector(
        imoveis_brutos, name='imoveis', target_crs=cfg.project.target_crs,
        minimum_area_m2=cfg.analysis.minimum_geometry_area_m2,
        id_field=cfg.properties.id_field,
    )
    vegetacao_limpa, qc_vegetacao = clean_vector(
        vegetacao_bruta, name='vegetacao_declarada', target_crs=cfg.project.target_crs,
        minimum_area_m2=cfg.analysis.minimum_geometry_area_m2,
        id_field=cfg.declared_native.id_field,
    )

    area_estudo = dissolve_study_area(area_limpa)
    controle_qualidade = pd.DataFrame([
        qc_area.as_record(), qc_imoveis.as_record(), qc_vegetacao.as_record()
    ])
    display(controle_qualidade)
else:
    print('Etapa aguardando os arquivos reais.')

## 4 Recorte pela bacia e inspeção cartográfica

As feições que apenas intersectam a bacia são recortadas pelo limite hidrográfico. O mapa abaixo é uma inspeção inicial e não substitui a análise dos atributos ou das sobreposições.

In [ ]:
if DADOS_DISPONIVEIS:
    imoveis = clip_to_study_area(imoveis_limpos, area_estudo)
    vegetacao = clip_to_study_area(vegetacao_limpa, area_estudo)
    print(f'Imóveis no recorte: {len(imoveis):,}')
    print(f'Polígonos de vegetação declarada: {len(vegetacao):,}')

    fig, ax = plt.subplots(figsize=(10, 8))
    area_estudo.boundary.plot(ax=ax, color='black', linewidth=1.5, label='Bacia do Gramame')
    imoveis.boundary.plot(ax=ax, color='#8c8c8c', linewidth=0.25, alpha=0.65, label='Imóveis CAR')
    vegetacao.plot(ax=ax, color='#238b45', edgecolor='none', alpha=0.70, label='Vegetação declarada')
    ax.set_title('Imóveis rurais e vegetação nativa declarada')
    ax.set_axis_off()
    ax.legend(loc='lower left')
    plt.show()
else:
    print('Mapa aguardando os arquivos reais.')

## 5 Indicadores vetoriais

O somatório das áreas declaradas pode contar duas vezes regiões sobrepostas. Por isso são apresentados tanto o somatório dos polígonos quanto a área de sua união geométrica.

In [ ]:
if DADOS_DISPONIVEIS:
    vegetacao_uniao = geometry_union(vegetacao, 'vegetacao_declarada_uniao')
    indicadores_vetoriais = pd.DataFrame([
        vector_indicators(imoveis, vegetacao, vegetacao_uniao)
    ]).T.rename(columns={0: 'valor'})
    display(indicadores_vetoriais)
else:
    print('Indicadores aguardando os arquivos reais.')

## 6 Cruzamento com o MapBiomas

O raster é lido em grade métrica com resolução configurada de 30 m e reamostragem pelo vizinho mais próximo. O índice principal é:

$$IC_{CAR-MB} = 100 \times \frac{A_{declarada \cap natural}}{A_{declarada,\ raster\ válido}}$$

A lista de códigos naturais é configurável e precisa ser registrada na metodologia final.

In [ ]:
if DADOS_DISPONIVEIS:
    resumo_raster, classes_mapbiomas = summarize_classes_in_geometry(
        cfg.mapbiomas.path,
        vegetacao_uniao,
        target_crs=cfg.project.target_crs,
        resolution_m=cfg.analysis.raster_resolution_m,
        natural_codes=cfg.analysis.natural_class_codes,
        all_touched=cfg.analysis.all_touched,
    )
    resumo = pd.Series({
        'área vetorial declarada (ha)': resumo_raster.geometry_area_ha,
        'área raster válida (ha)': resumo_raster.valid_area_ha,
        'área natural coincidente (ha)': resumo_raster.natural_area_ha,
        'área não natural (ha)': resumo_raster.non_natural_area_ha,
        'área sem dado (ha)': resumo_raster.nodata_area_ha,
        'índice de convergência (%)': resumo_raster.convergence_percent,
    }, name='valor')
    display(resumo.to_frame())
    display(classes_mapbiomas)
else:
    print('Cruzamento raster-vetor aguardando os arquivos reais.')

In [ ]:
if DADOS_DISPONIVEIS:
    cores = classes_mapbiomas['is_natural'].map({True: '#238b45', False: '#d95f0e'})
    ax = classes_mapbiomas.plot.bar(
        x='class_code', y='area_ha', color=cores, legend=False, figsize=(11, 5)
    )
    ax.set_title('Classes do MapBiomas dentro da vegetação declarada')
    ax.set_xlabel('Código da classe')
    ax.set_ylabel('Área (ha)')
    plt.tight_layout()
    plt.show()
else:
    print('Gráfico aguardando os arquivos reais.')

## 7 Execução completa e exportação

A execução abaixo repete as etapas por meio do pipeline validado e grava GeoPackage, indicadores, controle de qualidade, metadados e log no diretório definido no YAML.

In [ ]:
if DADOS_DISPONIVEIS:
    diretorio_resultados = cfg.project.output_dir
    arquivo_indicadores = diretorio_resultados / 'indicadores_gerais.csv'
    if not arquivo_indicadores.exists():
        diretorio_resultados = run_pipeline(CONFIG_PATH)
        print(f'Resultados gerados em: {diretorio_resultados}')
    else:
        print(f'Reutilizando resultados já processados em: {diretorio_resultados}')

    indicadores_finais = pd.read_csv(arquivo_indicadores)
    qualidade_final = pd.read_csv(diretorio_resultados / 'controle_qualidade.csv')
    display(indicadores_finais.T.rename(columns={0: 'valor'}))
    display(qualidade_final)
else:
    print('Exportação não executada porque ainda faltam entradas.')

In [ ]:
if DADOS_DISPONIVEIS:
    figuras = sorted((diretorio_resultados / 'figuras').glob('*.png'))
    print(f'{len(figuras)} mapas em PNG foram gerados:')
    for figura in figuras:
        print(figura.name)
        display(NotebookImage(filename=str(figura), width=900))
else:
    print('Pré-visualização dos mapas aguardando os arquivos reais.')

## 8 Registro para interpretação

Antes de usar os resultados, registrar:

- data e edição de cada base;
- lista final de classes naturais do MapBiomas;
- número e tratamento de geometrias inválidas e duplicadas;
- diferença entre somatório declaratório e união geométrica;
- área sem dados válidos no raster;
- sensibilidade ao parâmetro `all_touched`;
- limitações decorrentes de escala, resolução, temporalidade e natureza autodeclaratória do CAR.